# HypotheSAEs Quickstart

This notebook demonstrates basic usage of HypotheSAEs on a sample of the Yelp review dataset.
By default we use `gpt-5.2` for hypothesis generation (interpreting neurons) and `gpt-5-mini` for text annotation.

Set `OPENAI_KEY_SAE` below for OpenAI-hosted requests. If you are using a local OpenAI-compatible endpoint (for example, vLLM server mode), set `OPENAI_BASE_URL`; the key is optional for non-OpenAI hosts.


In [25]:
%load_ext autoreload
%autoreload 2

import os
os.environ['OPENAI_KEY_SAE'] = '...'  # Replace with your OpenAI API key (or set from another env var)
os.environ['OPENAI_BASE_URL'] = 'http://0.0.0.0:8001/v1'  # Optional: local OpenAI-compatible endpoint

import pandas as pd

import importlib
import hypothesaes.quickstart
importlib.reload(hypothesaes.quickstart)
from hypothesaes.quickstart import interpret_sae, interpret_sep_sae

INTERPRETER_MODEL = "openai/gpt-oss-20b"
ANNOTATOR_MODEL = INTERPRETER_MODEL

# Optional request kwargs forwarded directly to the Responses API.
# Keep empty by default; set only what you need at call time.
# Example: INTERPRET_LLM_KWARGS = {"reasoning_effort": "low"}
INTERPRET_LLM_KWARGS = {"reasoning_effort": "low"}
ANNOTATION_LLM_KWARGS = {"reasoning_effort": "low"}

N_WORKERS_ANNOTATION = 10  # Lower if hitting API rate limits


Failed to reload module 'hypothesaes' from file '/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/hypothesaes/__init__.py'
Traceback (most recent call last):
  File "/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 326, in check
    superreload(m, reload, self.old_objects)
  File "/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 580, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/blue/bonniejdorr/ethanlmines/uv/python/cpython-3.12.13-linux-x86_64-gnu/lib/python3.12/importlib/__init__.py", line 131, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 866, in _exec
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/hyp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of hypothesaes failed: Traceback (most recent call last):
  File "/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 326, in check
    superreload(m, reload, self.old_objects)
  File "/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 580, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/blue/bonniejdorr/ethanlmines/uv/python/cpython-3.12.13-linux-x86_64-gnu/lib/python3.12/importlib/__init__.py", line 131, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 866, in _exec
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/hypothesaes/__init__.py", line 4, in <module>
    from .quickstart import (
ImportError: c

**Load data**

The dataset we're using here is a subset of 20K Yelp reviews, with 2K reviews used for validation (during SAE training). 

The target variable is the `stars` column, which is a rating between 1 and 5. We treat this as a regression task.

There are also 2K reviews used for holdout evaluation, which we'll use at the end of the notebook.

In [11]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"
data_path = os.path.join(prefix, 'data', 'vast_train.csv')
data_path = os.path.join(prefix, 'data', 'raw_train_all_onecol.csv')
is_vast = 'vast' in data_path
full_df = pd.read_csv(data_path)

In [12]:
import random
rng = random.Random(0)
indices = list(range(len(full_df)))
rng.shuffle(indices)

In [13]:
train_stop = int(len(full_df) * .75)
train_df = full_df.iloc[indices[:train_stop]]
val_df = full_df.iloc[indices[train_stop:]]

In [14]:
len(train_df), len(val_df), len(full_df)

(10317, 3439, 13756)

In [15]:
if is_vast:
    DOC_FIELD = 'post'
    TOPIC_FIELD = 'new_topic'
else:
    DOC_FIELD = "Ori Text"
    TOPIC_FIELD = "Target 1"
train_texts, train_topics = list(train_df[DOC_FIELD]), list(train_df[TOPIC_FIELD])
val_texts, val_topics     = list(val_df[DOC_FIELD])  , list(val_df[TOPIC_FIELD])

In [16]:
from transformers import AutoConfig
embed_name = "nomic-ai/modernbert-embed-base"
embed_model_config = AutoConfig.from_pretrained(embed_name)

In [8]:
from hypothesaes.sae import SparseAutoencoder
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
class CrossSae(SparseAutoencoder):
    def __init__(self,
                 **kwargs):
        embedder = AutoModelForSequenceClassification.from_pretrained(embed_name).eval()
        embed_dim = embedder.config.hidden_size
        print(f"embed_dim={embed_dim}")
        super().__init__(input_dim=2 * embed_dim, **kwargs)
        self.embed_dim = embed_dim
        self.embedder: torch.nn.Module = embedder
        self.tokenizer = AutoTokenizer.from_pretrained(embed_name)
        for w in self.embedder.parameters():
            w.requires_grad_ = False

        self.film_gate = torch.nn.Sequential(
            torch.nn.Linear(embed_dim, embed_dim, bias=True),
            torch.nn.Tanh()
        )
        self.film_bias = torch.nn.Linear(embed_dim, embed_dim)
        # self.post_tran = torch.nn.Linear(embed_dim, 2 * embed_dim)

    @torch.inference_mode
    def preprocess(self, targets: list[str], text: list[str], batch_size: int = 16):
        if len(targets) != len(text):
            raise ValueError("targets and text must have the same length")
        if batch_size < 1:
            raise ValueError("batch_size must be positive")

        device = next(self.embedder.parameters()).device
        if not targets:
            return torch.empty((0, 2 * self.embed_dim), device=device)

        def embed_batch(items: list[str]):
            tokenized = self.tokenizer(
                items,
                padding=False,
                truncation=True,
                return_attention_mask=True,
            )
            sorted_indices = sorted(
                range(len(items)),
                key=lambda index: len(tokenized["input_ids"][index]),
            )
            dataloader = DataLoader(
                sorted_indices,
                batch_size=batch_size,
                shuffle=False,
                collate_fn=lambda indices: self.tokenizer.pad(
                    {
                        key: [tokenized[key][index] for index in indices]
                        for key in tokenized
                    },
                    return_tensors="pt",
                ),
                pin_memory=device.type == "cuda",
            )

            sorted_embeddings = []
            for batch in tqdm(dataloader, desc="Embedding batches"):
                sorted_embeddings.extend(self._embed(
                    batch.to(device, non_blocking=device.type == "cuda")
                ))

            embeddings = [None] * len(items)
            for index, embedding in zip(sorted_indices, sorted_embeddings):
                embeddings[index] = embedding
            return torch.stack(embeddings)

        text_embeddings = embed_batch(list(text))
        target_embeddings = embed_batch(list(targets))
        return torch.cat((target_embeddings, text_embeddings), dim=-1).detach().cpu()

    def _embed(self, doc_ids):
        return self.embedder.base_model(**doc_ids).last_hidden_state[:, 0]

    def forward(self, x: torch.Tensor):
        A = x[..., :self.embed_dim]
        B = x[..., self.embed_dim:]
        mixed = self.film_gate(A) * B + self.film_bias(A)
        recon, info = super().forward( torch.cat([A, mixed], dim=1) )
        return recon, info


In [9]:
from hypothesaes.embedding import get_joint_stance_embeddings
train_embeddings = get_joint_stance_embeddings(train_topics, train_texts)
val_embeddings = get_joint_stance_embeddings(val_topics, val_texts)

Invalid model-index. Not loading eval results into CardData.


Invalid model-index. Not loading eval results into CardData.
Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Embedding target/post pairs:   0%|          | 0/1264 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [21]:
from hypothesaes.sae import SepSae
sae = SepSae(
    m_total_neurons=256, k1=8, k2=8
)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: nomic-ai/modernbert-embed-base
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
head.norm.weight  | MISSING | 
head.dense.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
train_embeddings = sae.preprocess(train_topics, train_texts, batch_size=16)
val_embeddings = sae.preprocess(val_topics, val_texts, batch_size=16)

Computing embeddings:   0%|          | 0/645 [00:00<?, ?it/s]

Computing embeddings:   0%|          | 0/645 [00:00<?, ?it/s]

Computing embeddings:   0%|          | 0/215 [00:00<?, ?it/s]

Computing embeddings:   0%|          | 0/215 [00:00<?, ?it/s]

In [22]:
train_top_embeds, train_text_embeds = train_embeddings
val_top_embeds, val_text_embeds     = val_embeddings
sae.fit(train_top_embeds, train_text_embeds, val_top_embeds, val_text_embeds, n_epochs=300)

  0%|          | 0/300 [00:00<?, ?it/s]

{'train_loss': [0.9437962855611529,
  0.8274743727275303,
  0.7248067742302304,
  0.6539911996750605,
  0.6055534113021124,
  0.5723421374956766,
  0.5455000201861063,
  0.5255623431432814,
  0.5094693530173529,
  0.4950110784598759,
  0.4852909190314157,
  0.47500334892954144,
  0.468046920640128,
  0.4597868223985036,
  0.45510500243731905,
  0.44925021983328317,
  0.4453327627409072,
  0.44146381673358737,
  0.4369598769006275,
  0.43314991394678753,
  0.4312870828878312,
  0.42803931094351266,
  0.42541649369966417,
  0.42345806672459557,
  0.4220334305649712,
  0.41935834288597107,
  0.41810856262842816,
  0.41562934148879277,
  0.41432818344661165,
  0.4137968094575973,
  0.4111958018371037,
  0.40996084610621136,
  0.40897704448018757,
  0.4077158726397015,
  0.4065933497179122,
  0.40591563355355037,
  0.40537464192935396,
  0.40355195317949566,
  0.40327256492206026,
  0.40175712676275344,
  0.4008840875966208,
  0.40068734117916655,
  0.3998472378367469,
  0.3992997961384909,

In [ ]:
# sae = CrossSae(m_total_neurons=256, k_active_neurons=8, prefix_lengths=[32, 256]).cuda()
# train_embeddings = sae.preprocess(train_topics, train_texts, batch_size=16)
# val_embeddings = sae.preprocess(val_topics, val_texts, batch_size=16)
# sae.fit(
#     train_embeddings,
#     val_embeddings,
# )

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: nomic-ai/modernbert-embed-base
Key               | Status  | 
------------------+---------+-
head.norm.weight  | MISSING | 
classifier.bias   | MISSING | 
classifier.weight | MISSING | 
head.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


embed_dim=768


**Interpret neurons**  

Interpret a random subset of neurons in the SAE to sanity-check that the learned features, and their interpretations, seem reasonable. We generate and print labels for `n_random_neurons` neurons, and we also print out the top-activating texts for each neuron.

In [23]:
texts_for_prompt = [f"Topic: {target} | Post: {text}" for target, text in zip(train_topics, train_texts)]

In [24]:
# This instruction will be included in the neuron interpretation prompt.
# The below instructions are specific to Yelp, but you can customize this for your task.
# If you don't pass in task-specific instructions, there is a generic instruction (see src/interpret_neurons.py);
# task-specific instructions are optional, but they help produce hypotheses at the desired level of specificity.

TASK_SPECIFIC_INSTRUCTIONS = """All of the below samples are (topic, post) pairs from an online forum.
Features should describe a specific aspect of how the post does (or does not) relate the topic.
Examples could be:
- 'praises another entity with positive associations to the topic'
- 'praises another entity with negative associations to the topic'
- 'criticizes another entity with negative associations to the topic'
- 'criticizes another entity with positive associations to the topic'

Focus on the relationship between the post and the topic. If it's possible to determine whether the feature applies by the post alone, you're doing it wrong."""

# Interpret random neurons
results = interpret_sae(
    texts=texts_for_prompt,
    embeddings=train_embeddings,
    sae=sae,
    n_random_neurons=5,
    print_examples_n=3,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
    interpreter_model=INTERPRETER_MODEL,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
)


Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Activations shape: (10317, 512)


Generating interpretations:   0%|          | 0/5 [00:00<?, ?it/s]


Neuron 133 (2.0% active): mentions the phrase 'climate change'

Top activating examples:
1. Topic: Climate Change | Post: I met with NATO Secretary General Stoltenberg today to discuss the upcoming Madrid Summit. The U.S. and our Allies will strengthen NATO’s deterrence and defense and tackle new challenges, from cyber to climate change. ALT 930 1,260 7,630
2. Topic: Climate Change | Post: Btw, "Nuclear Energy" is literally the easiest solution to combat Climate Change, and governments worldwide oppose it or prohibit it. Rich people and multi-million dollar companies would use it for being more efficient and cheaper, but hey! Let's not ruin the Leftist Narrative  1 3 73
3. Topic: Climate Change | Post: People who don’t think a period of drastic collapse in the imperial core is happening soon are apparently unaware of the house of cards of Western neocolonial exploitation and how climate change is already causing famines due to crop failures. Make no mistake: we are next, soon. 5 12 70

In [26]:
# Interpret random neurons
results = interpret_sep_sae(
    topics=train_topics,
    texts=train_texts,
    sae=sae,
    n_random_neurons=5,
    print_examples_n=3,
    interpreter_model=INTERPRETER_MODEL,
    interpret_llm_kwargs=INTERPRET_LLM_KWARGS,
)

Computing embeddings:   0%|          | 0/645 [00:00<?, ?it/s]

Computing embeddings:   0%|          | 0/645 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Topics activations shape: (10317, 256)


Generating interpretations:   0%|          | 0/5 [00:00<?, ?it/s]


Topics neuron 72 (7.9% active): mentions FIFA

Top activating examples:
1. FIFA
2. FIFA
3. FIFA
----------------------------------------------------------------------------------------------------

Topics neuron 189 (3.3% active): mentions climate

Top activating examples:
1. climate
2. climate
3.  climate
----------------------------------------------------------------------------------------------------

Topics neuron 132 (3.7% active): mentions EU

Top activating examples:
1. Europe
2. Europe
3. Europe
----------------------------------------------------------------------------------------------------

Topics neuron 82 (2.3% active): mentions nuclear energy

Top activating examples:
1. Nuclear
2. Nuclear
3. Nuclear energy
----------------------------------------------------------------------------------------------------

Topics neuron 99 (2.8% active): mentions Russia

Top activating examples:
1. Russia
2. Russia
3. Russia
----------------------------------------------------------

Generating interpretations:   0%|          | 0/5 [00:00<?, ?it/s]


Texts neuron 238 (1.8% active): contains three consecutive integer tokens each on its own line at the end of the text

Top activating examples:
1. AMD 4700S: "We expect to see over 80 designs come to market from our SI [System Integrator] partners beginning on June 24th. Prices of these systems will be announced by our SI partners in due course."  Wow, that is not bad 4 11 63
2. AMD 4700S: "We expect to see over 80 designs come to market from our SI [System Integrator] partners beginning on June 24th. Prices of these systems will be announced by our SI partners in due course."  Wow, that is not bad 4 11 63
3. Ultratech announced a capex of 12k crs   Reliance has acquired 11 cos in energy space & capex of $10Bn+ in clean energy.  Divis is doing more capex in next 3 yrs than last 30yrs  Mkt leaders across sectors r investing for future  Don’t lose sight of d big picture during downturns 14 65 829
-------------------------------------------------------------------------------------------

In [29]:
results['topics']

,neuron_idx,interpretation,activation_percent,top_example_1,top_example_2,top_example_3
0,72,mentions FIFA,7.889890,FIFA,FIFA,FIFA
1,189,mentions climate,3.285839,climate,climate,climate
2,132,mentions EU,3.654163,Europe,Europe,Europe
3,82,mentions nuclear energy,2.268101,Nuclear,Nuclear,Nuclear energy
4,99,mentions Russia,2.752738,Russia,Russia,Russia


In [30]:
results['texts']

,neuron_idx,interpretation,activation_percent,top_example_1,top_example_2,top_example_3
0,238,contains three consecutive integer tokens each...,1.754386,"AMD 4700S:\n""We expect to see over 80 designs ...","AMD 4700S:\n""We expect to see over 80 designs ...",Ultratech announced a capex of 12k crs \n\nRel...
1,65,mentions Trump,2.965978,0 - Stars High School Senior Year\n1 - Colleg...,0 - Stars High School Senior Year\n1 - Colleg...,10% for the big guy. How much was sent to Ukra...
2,189,mentions Pride,2.258408,Pfizer says it s completed its rolling submiss...,Even Minorities are the Citizen of this Countr...,Even Minorities are the Citizen of this Countr...
3,19,starts with a number followed by a space,3.818940,2) I’ve written up the contribution to the fie...,"4\n... than we ever were to opioids, the impac...","4\n... than we ever were to opioids, the impac..."
4,99,contains a hashtag that starts with a capital ...,2.306872,please all International student / Dependent v...,please all International student / Dependent v...,"The Weekly World News just called, they want t..."
